# Training wake-word models for Nova (experimental)

This notebook trains two small models with [openWakeWord](https://github.com/dscripka/openWakeWord): one for **"hey nova"** and one for **"nova"**. It is an experiment: the results are scored against Nova's test set before anything is put into the app.

**Before you start**

1. Menu: **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells from top to bottom, one at a time. Wait for each to finish.
3. The whole thing takes roughly two to three hours. Keep the tab open.

**What data this uses, and where it goes**

- Everything runs on a temporary Google computer that is erased when the session ends.
- **No recording of your voice is used or uploaded.** The examples of "hey nova" are made by a text-to-speech program (Piper) inside this notebook.
- It downloads public research datasets: room echoes (MIT), background sounds (AudioSet), music (Free Music Archive), and features of about 2,000 hours of general speech (ACAV100M, 17 GB).
- At the end it gives you one small file to download, `nova-wake-models.zip`.

This notebook is adapted from openWakeWord's own `automatic_model_training.ipynb`. It has not been run by its author here, so cell 4 does a two-minute trial first to catch any problem early.

In [ ]:
# 1. Set up the tools (about 3 minutes)
!git clone -q https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize webrtcvad
!git clone -q https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword
!pip install -q mutagen torchinfo torchmetrics speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics pronouncing datasets==2.14.6 deep-phonemizer==0.0.19 onnx onnxruntime pyyaml
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
for f in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    os.system(f"wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{f} -O ./openwakeword/openwakeword/resources/models/{f}")
import torch
print("GPU ready:", torch.cuda.is_available(), "  <- this must say True. If it says False, set the runtime to T4 GPU and start again.")

In [ ]:
# 2. Download the public datasets (about 15 to 25 minutes; the big one is 17 GB)
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
from tqdm import tqdm

os.makedirs("mit_rirs", exist_ok=True)
for row in tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)):
    scipy.io.wavfile.write(os.path.join("mit_rirs", row["audio"]["path"].split("/")[-1]), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

os.makedirs("audioset", exist_ok=True); os.makedirs("audioset_16k", exist_ok=True)
!wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
!cd audioset && tar -xf bal_train09.tar
ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]}).cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(ds):
    scipy.io.wavfile.write(os.path.join("audioset_16k", row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

os.makedirs("fma", exist_ok=True)
fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True).cast_column("audio", datasets.Audio(sampling_rate=16000)))
for i in tqdm(range(120)):      # about an hour of music
    row = next(fma)
    scipy.io.wavfile.write(os.path.join("fma", row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Downloads finished.")

In [ ]:
# 3. The settings shared by both models
import yaml, sys, os
# Words and phrases the models must learn NOT to wake on. These are the ones Nova's tests found troublesome.
NOT_THE_NAME = ["no", "no.", "now", "know", "nope", "no way", "noah", "novel", "november", "nobody", "over", "innovate", "innovation",
                "nevada", "lava", "mocha", "nola", "know that", "no thanks", "not now", "hey no", "hey now", "hey noah", "hey you", "a nova scotia",
                "no the", "know the", "nava", "never", "navy", "hey nobody", "hey november"]

def make_config(phrase, name, samples, steps):
    c = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
    c["target_phrase"] = [phrase]
    c["model_name"] = name
    c["custom_negative_phrases"] = [p for p in NOT_THE_NAME if p != phrase]
    c["n_samples"] = samples
    c["n_samples_val"] = max(200, samples // 5)
    c["steps"] = steps
    c["background_paths"] = ["./audioset_16k", "./fma"]
    c["background_paths_duplication_rate"] = [1, 1]
    c["false_positive_validation_data_path"] = "validation_set_features.npy"
    c["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
    c["output_dir"] = f"./models_{name}"
    with open(f"{name}.yaml", "w") as f: yaml.dump(c, f)
    return f"{name}.yaml"

def train(phrase, name, samples, steps):
    cfg = make_config(phrase, name, samples, steps)
    for stage in ["--generate_clips", "--augment_clips", "--train_model"]:
        code = os.system(f"{sys.executable} openwakeword/openwakeword/train.py --training_config {cfg} {stage}")
        print(stage, "finished with code", code)
    out = f"./models_{name}/{name}.onnx"
    print("MODEL SAVED:" if os.path.exists(out) else "NO MODEL WAS PRODUCED:", out)
    return os.path.exists(out)
print("Settings ready.")

In [ ]:
# 4. A two-minute trial with a tiny amount of data, to make sure every stage works before the long runs.
#    If this prints "NO MODEL WAS PRODUCED", stop here and send the output of this cell back.
ok = train("hey nova", "trial_run", samples=120, steps=300)
print("The trial worked. Carry on with the next cell." if ok else "The trial failed. Do not run the next cells.")

In [ ]:
# 5. Train "hey nova" (roughly an hour)
train("hey nova", "hey_nova", samples=6000, steps=25000)

In [ ]:
# 6. Train "nova" on its own (roughly an hour)
train("nova", "nova", samples=6000, steps=25000)

In [ ]:
# 7. Package the two models and download them
import shutil, os
os.makedirs("nova-wake-models", exist_ok=True)
for n in ["hey_nova", "nova"]:
    p = f"./models_{n}/{n}.onnx"
    if os.path.exists(p): shutil.copy(p, "nova-wake-models/")
    else: print("missing:", p)
shutil.make_archive("nova-wake-models", "zip", "nova-wake-models")
from google.colab import files
files.download("nova-wake-models.zip")

## Afterwards

Put `nova-wake-models.zip` in your Downloads folder and say it is there. The models are then scored on Nova's test recordings, and only if the numbers are good is anything tried with a real microphone.

**Licence note.** openWakeWord's code is Apache 2.0. The general-speech features used here come from datasets whose own licences are unclear, which is why openWakeWord publishes its ready-made models as non-commercial, share-alike (CC BY-NC-SA 4.0). Models trained here should be treated the same way.